# IAA Hackathon 2026 — Notebook 03: GLM & Blend

Fits a Tweedie GLM (statsmodels, power=1.5) using features selected from XGBoost importance. Uses target encoding with Bayesian shrinkage for categoricals. Blends XGBoost and GLM predictions. Generates submission files.

In [0]:
# Step 1: Load data, XGBoost importance, and XGBoost validation predictions
import pandas as pd
import numpy as np
import os

%pip install statsmodels openpyxl -q

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load processed training data
train = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    print(f'{line}: {train[line].shape}, target mean={train[line]["ultimate_claims"].mean():.4f}')

# Load test files
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))

# Load enrichment files
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Load XGBoost feature importance from Excel
EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_v4/'
xgb_importance = {}
for line in ['motor', 'buildings', 'contents']:
    sheet_name = f'{line} importance'
    xgb_importance[line] = pd.read_excel(os.path.join(EXCEL_DIR, 'xgboost_v4_results.xlsx'), sheet_name=sheet_name)
    xgb_importance[line] = xgb_importance[line].sort_values('importance', ascending=False)
    print(f'{line} XGBoost importance: top 5 = {xgb_importance[line]["feature"].head(5).tolist()}')

# Load XGBoost validation predictions (for blend D2)
xgb_val_preds = {}
for line in ['motor', 'buildings', 'contents']:
    xgb_val_preds[line] = pd.read_parquet(os.path.join(PROC_DIR, f'xgb_val_pred_{line}.parquet'))
    print(f'{line} XGB val preds: {len(xgb_val_preds[line])} rows')

# Load XGBoost test predictions from submission CSV
xgb_test_preds = {}
xgb_sub = pd.read_csv('/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/submissions_v4/xgboost_tweedie_v4.csv')
for line in ['motor', 'buildings', 'contents']:
    prefix = line[0].upper()  # M, B, C
    mask = xgb_sub['row_id'].str.startswith(prefix)
    xgb_test_preds[line] = dict(zip(xgb_sub.loc[mask, 'row_id'], xgb_sub.loc[mask, 'prediction']))
    print(f'{line} XGB test preds: {len(xgb_test_preds[line])} rows')

# XGBoost validation D2 scores for reference (updated with Year feature for buildings/contents)
xgb_d2 = {'motor': 0.0599, 'buildings': 0.0588, 'contents': 0.0470}

# Trend factors for 2025 predictions (from corrected BF development factors)
# Motor: CAGR 2021-2023 ~3.7% (severity-driven, frequency declining)
# Buildings: CAGR 2021-2024 = 9.6% (very consistent)
# Contents: CAGR 2021-2024 = 4.9%
TREND_FACTORS = {
    'motor': 1.04,
    'buildings': 1.096,
    'contents': 1.049,
}
print(f'\nTrend uplift factors for 2025 (XGBoost): {TREND_FACTORS}')

# GLM trend factors: GLM now has Year as a feature, so it extrapolates the trend itself.
# For buildings/contents, Year captures the full trend — no extra uplift needed.
# For motor, the 2024 claims dip dampens the Year coefficient,
# so we still apply the explicit 4% uplift.
GLM_TREND_FACTORS = {
    'motor': 1.04,
    'buildings': 1.0,
    'contents': 1.0,
}
print(f'GLM trend factors for 2025: {GLM_TREND_FACTORS}')

In [0]:
# Step 2: Prepare test data (clean + join enrichment — same as Notebook 2)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

test['buildings']['construction_type'] = test['buildings']['construction_type'].astype(str).str.strip().str.upper().replace({'W': 'TIMBER', 'T': 'TIMBER'})
test['motor']['customer_id'] = test['motor']['customer_id'].str.zfill(8)
customers['customer_id'] = customers['customer_id'].astype(str).str.zfill(8)

for line in ['motor', 'buildings', 'contents']:
    if 'compulsory_excess' in test[line].columns:
        test[line].drop('compulsory_excess', axis=1, inplace=True)

customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df
    print(f'test_{line}: {df.shape}')

In [0]:
# Step 3: Fit Tweedie GLM using XGBoost top features
# Use target encoding (Bayesian shrinkage) for ALL categoricals — avoids convergence issues
# from one-hot encoding while including as many features as possible

import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

SMOOTHING = 30
MAX_FEATS = 25  # top N features by XGBoost importance

def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

def target_encode(train_series, train_target, apply_series, smoothing=30):
    """Bayesian target encoding: smooth category means toward prior."""
    prior = train_target.mean()
    grp = pd.DataFrame({'cat': train_series, 'target': train_target})
    stats = grp.groupby('cat')['target'].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + prior * smoothing) / (stats['count'] + smoothing)
    train_enc = train_series.map(smooth).fillna(prior)
    apply_enc = apply_series.map(smooth).fillna(prior)
    return train_enc, apply_enc

glm_models = {}
glm_val_scores = {}
glm_val_preds = {}
glm_test_preds = {}
glm_coefs = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}')
    print(f'=== {line.upper()} GLM ===')
    print(f'{"="*60}')
    
    train_df = train[line]
    test_df = test[line]
    
    # Select top MAX_FEATS features by XGBoost importance
    top_feats = xgb_importance[line]['feature'].head(MAX_FEATS).tolist()
    # Filter to features that exist in data
    top_feats = [f for f in top_feats if f in train_df.columns and f not in ['ultimate_claims', 'row_id', 'policy_id', 'customer_id', 'address_id', 'period_start', 'period_end']]
    # Force Year into GLM features (captures trend/inflation, GLM can extrapolate to 2025)
    if 'Year' not in top_feats and 'Year' in train_df.columns:
        top_feats.append('Year')
        print(f'  -> Forced Year into GLM features for trend capture')
    print(f'  Selected {len(top_feats)} features from XGBoost importance')
    
    # Time-based split
    year = train_df['Year'].values
    train_mask = year <= 2023
    val_mask = year == 2024
    
    y_tr = train_df.loc[train_mask, 'ultimate_claims'].values
    y_val = train_df.loc[val_mask, 'ultimate_claims'].values
    exp_tr = train_df.loc[train_mask, 'exposure'].values
    exp_val = train_df.loc[val_mask, 'exposure'].values
    exp_test = test_df['exposure'].values
    
    # Prepare features: target encode categoricals, standardize numerics
    X_tr = pd.DataFrame(index=train_df.index[train_mask])
    X_val = pd.DataFrame(index=train_df.index[val_mask])
    X_test = pd.DataFrame(index=test_df.index)
    
    for col in top_feats:
        tr_vals = train_df.loc[train_mask, col]
        val_vals = train_df.loc[val_mask, col]
        test_vals = test_df[col]
        
        is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'
        
        if is_cat:
            # Target encode (using training data only)
            tr_enc, val_enc = target_encode(tr_vals, y_tr, val_vals, SMOOTHING)
            _, test_enc = target_encode(tr_vals, y_tr, test_vals, SMOOTHING)
            X_tr[col] = tr_enc.values
            X_val[col] = val_enc.values
            X_test[col] = test_enc.values
        else:
            # Numeric: fill NaN with median, standardize
            med = tr_vals.median()
            X_tr[col] = tr_vals.fillna(med).values
            X_val[col] = val_vals.fillna(med).values
            X_test[col] = test_vals.fillna(med).values
    
    # Standardize all features
    scaler = StandardScaler()
    X_tr_scaled = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    
    # Add intercept
    X_tr_scaled = sm.add_constant(X_tr_scaled, has_constant='add')
    X_val_scaled = sm.add_constant(X_val_scaled, has_constant='add')
    X_test_scaled = sm.add_constant(X_test_scaled, has_constant='add')
    
    print(f'  Train: {X_tr_scaled.shape}, Val: {X_val_scaled.shape}')
    
    # Fit GLM Tweedie with log(exposure) offset
    try:
        model = sm.GLM(y_tr, X_tr_scaled,
                       family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
                       offset=np.log(exp_tr + 1e-10))
        result = model.fit(max_iter=100)
        glm_models[line] = result
        print(f'  Converged: {result.converged}')
        
        # Validate
        val_pred = result.predict(X_val_scaled, offset=np.log(exp_val + 1e-10))
        val_pred = np.maximum(val_pred, 0)
        d2 = tweedie_d2(y_val, val_pred, p=1.5)
        glm_val_scores[line] = d2
        glm_val_preds[line] = val_pred
        print(f'  Validation D2: {d2:.4f} (XGBoost: {xgb_d2[line]:.4f})')
        print(f'  Val pred mean: {val_pred.mean():.4f} (actual: {y_val.mean():.4f})')
        
        # Coefficients
        coefs = pd.DataFrame({'feature': X_tr_scaled.columns, 'coef': result.params, 'pvalue': result.pvalues})
        coefs = coefs.sort_values('coef', ascending=False)
        glm_coefs[line] = coefs
        print(f'  Top 5 coefficients:')
        print(f'  {coefs.head(5).to_string(index=False)}')
        
        # Predict on test
        test_pred = result.predict(X_test_scaled, offset=np.log(exp_test + 1e-10))
        test_pred = np.maximum(test_pred, 0.01)
        glm_test_preds[line] = test_pred
        print(f'  Test predictions: mean={test_pred.mean():.4f}, min={test_pred.min():.4f}, max={test_pred.max():.4f}')
        
    except Exception as e:
        print(f'  GLM FAILED: {e}')
        glm_val_scores[line] = None
        glm_test_preds[line] = np.full(len(test_df), test_df['exposure'].values * y_tr.mean())

print(f'\n{"="*60}')
print(f'=== Summary: GLM Validation D2 Scores ===')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: GLM={glm_val_scores[line]}, XGB={xgb_d2[line]}')

In [0]:
# Step 4: Blend XGBoost and GLM, generate submissions
SUB_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/submissions_v4/'

# Blend on validation to get D2
blend_val_d2 = {}
for line in ['motor', 'buildings', 'contents']:
    xgb_vp = xgb_val_preds[line]['xgb_pred'].values
    glm_vp = glm_val_preds[line]
    y_val = xgb_val_preds[line]['ultimate_claims'].values
    
    # 50/50 blend
    blend50 = 0.5 * xgb_vp + 0.5 * glm_vp
    d2_50 = tweedie_d2(y_val, blend50, p=1.5)
    
    # 70/30 blend (XGBoost weight = 0.7)
    blend70 = 0.7 * xgb_vp + 0.3 * glm_vp
    d2_70 = tweedie_d2(y_val, blend70, p=1.5)
    
    blend_val_d2[line] = {'5050': d2_50, '7030': d2_70}
    print(f'{line}: XGB={xgb_d2[line]:.4f}, GLM={glm_val_scores[line]:.4f}, Blend50/50={d2_50:.4f}, Blend70/30={d2_70:.4f}')

# Generate blended test predictions and save submissions
for blend_name, xgb_w in [('5050', 0.5), ('7030', 0.7)]:
    submission = sample_submission.copy()
    for line in ['motor', 'buildings', 'contents']:
        test_df = test[line]
        glm_pred = np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01)
        glm_pred = np.clip(glm_pred, 0.01, 50000)
        # Apply trend uplift to GLM 2025 predictions (motor only — buildings/contents have Year in GLM)
        glm_pred = glm_pred * GLM_TREND_FACTORS[line]
        # Get XGBoost test predictions for this line (already has trend uplift from submission CSV)
        xgb_pred = np.array([xgb_test_preds[line].get(rid, 0.01) for rid in test_df['row_id']])
        
        # Blend
        blend_pred = xgb_w * xgb_pred + (1 - xgb_w) * glm_pred
        blend_pred = np.clip(blend_pred, 0.01, 50000)
        
        pred_map = dict(zip(test_df['row_id'], blend_pred))
        mask = submission['row_id'].isin(pred_map)
        submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
    
    submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
    # Replace any remaining inf/nan
    submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
    sub_path = os.path.join(SUB_DIR, f'blend_{blend_name}_v4.csv')
    submission.to_csv(sub_path, index=False)
    print(f'\nSaved: {sub_path} (mean={submission["prediction"].mean():.4f})')

# Also save GLM-only submission (with trend uplift)
glm_sub = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    glm_pred = np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01)
    glm_pred = np.clip(glm_pred, 0.01, 50000) * GLM_TREND_FACTORS[line]
    pred_map = dict(zip(test_df['row_id'], glm_pred))
    mask = glm_sub['row_id'].isin(pred_map)
    glm_sub.loc[mask, 'prediction'] = glm_sub.loc[mask, 'row_id'].map(pred_map)
glm_sub['prediction'] = glm_sub['prediction'].clip(lower=0.01, upper=50000)
glm_sub['prediction'] = glm_sub['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
glm_sub.to_csv(os.path.join(SUB_DIR, 'glm_tweedie_v4.csv'), index=False)
print(f'Saved: glm_tweedie_v4.csv (mean={glm_sub["prediction"].mean():.4f})')

In [0]:
# Step 5: Excel checkpoint — save all model comparison and GLM coefficients
with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'glm_blend_v4_results.xlsx'), engine='openpyxl') as writer:
    # Sheet 1: Model comparison
    comparison = pd.DataFrame([
        {'line': line, 'xgb_d2': xgb_d2[line], 'glm_d2': glm_val_scores[line],
         'blend_5050': blend_val_d2[line]['5050'], 'blend_7030': blend_val_d2[line]['7030']}
        for line in ['motor', 'buildings', 'contents']
    ])
    comparison.loc['average'] = ['average', comparison['xgb_d2'].mean(), comparison['glm_d2'].mean(),
                                  comparison['blend_5050'].mean(), comparison['blend_7030'].mean()]
    comparison.to_excel(writer, sheet_name='Model Comparison', index=False)
    
    # Sheets 2-4: GLM coefficients per line
    for line in ['motor', 'buildings', 'contents']:
        if line in glm_coefs:
            glm_coefs[line].to_excel(writer, sheet_name=f'{line} coefs', index=False)

print('Saved: glm_blend_v4_results.xlsx')
print('\n=== Final Model Comparison ===')
print(f'{"line":12s} {"XGB":>8s} {"GLM":>8s} {"50/50":>8s} {"70/30":>8s}')
for line in ['motor', 'buildings', 'contents']:
    print(f'{line:12s} {xgb_d2[line]:8.4f} {glm_val_scores[line]:8.4f} {blend_val_d2[line]["5050"]:8.4f} {blend_val_d2[line]["7030"]:8.4f}')
avg_xgb = np.mean(list(xgb_d2.values()))
avg_glm = np.mean([v for v in glm_val_scores.values() if v is not None])
avg_50 = np.mean([blend_val_d2[l]['5050'] for l in ['motor', 'buildings', 'contents']])
avg_70 = np.mean([blend_val_d2[l]['7030'] for l in ['motor', 'buildings', 'contents']])
print(f'{"AVERAGE":12s} {avg_xgb:8.4f} {avg_glm:8.4f} {avg_50:8.4f} {avg_70:8.4f}')

In [0]:
# Step 5b: Optimize blend weights
# Search XGBoost weight from 0.20 to 0.80 in 0.05 steps

print('=== Optimizing Blend Weights ===\n')

best_weights = {}
best_d2 = {}

for line in ['motor', 'buildings', 'contents']:
    xgb_vp = xgb_val_preds[line]['xgb_pred'].values
    glm_vp = glm_val_preds[line]
    y_val = xgb_val_preds[line]['ultimate_claims'].values
    
    results = []
    for w in np.arange(0.20, 0.81, 0.05):
        blend = w * xgb_vp + (1 - w) * glm_vp
        d2 = tweedie_d2(y_val, blend, p=1.5)
        results.append((w, d2))
    
    results.sort(key=lambda x: -x[1])
    best_w = results[0][0]
    best_d2_val = results[0][1]
    
    best_weights[line] = best_w
    best_d2[line] = best_d2_val
    
    print(f'{line}: best weight = {best_w:.2f} (XGB), D2 = {best_d2_val:.4f}')
    for w, d2 in sorted(results, key=lambda x: x[0]):
        marker = ' <-- BEST' if abs(w - best_w) < 0.001 else ''
        print(f'  w={w:.2f}: D2={d2:.4f}{marker}')
    print()

# Overall optimal (single weight for all lines)
print('=== Overall Optimal (single weight, average D2) ===')
all_results = []
for w in np.arange(0.20, 0.81, 0.05):
    avg_d2 = np.mean([tweedie_d2(
        xgb_val_preds[line]['ultimate_claims'].values,
        w * xgb_val_preds[line]['xgb_pred'].values + (1 - w) * glm_val_preds[line],
        p=1.5
    ) for line in ['motor', 'buildings', 'contents']])
    all_results.append((w, avg_d2))

all_results.sort(key=lambda x: -x[1])
overall_best_w = all_results[0][0]
overall_best_d2 = all_results[0][1]
print(f'Overall best single weight = {overall_best_w:.2f}, avg D2 = {overall_best_d2:.4f}')
for w, d2 in sorted(all_results, key=lambda x: x[0]):
    marker = ' <-- BEST' if abs(w - overall_best_w) < 0.001 else ''
    print(f'  w={w:.2f}: avg D2={d2:.4f}{marker}')

# Per-line optimal average
per_line_avg = np.mean([best_d2[l] for l in ['motor', 'buildings', 'contents']])
print(f'\nPer-line optimal avg D2: {per_line_avg:.4f}')
print(f'Overall single-weight avg D2: {overall_best_d2:.4f}')
print(f'Best per-line weights: {best_weights}')

# Generate optimal submissions
# 1. Overall best single weight
submission = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    glm_pred = np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01)
    glm_pred = np.clip(glm_pred, 0.01, 50000) * GLM_TREND_FACTORS[line]
    xgb_pred = np.array([xgb_test_preds[line].get(rid, 0.01) for rid in test_df['row_id']])
    blend_pred = overall_best_w * xgb_pred + (1 - overall_best_w) * glm_pred
    blend_pred = np.clip(blend_pred, 0.01, 50000)
    pred_map = dict(zip(test_df['row_id'], blend_pred))
    mask = submission['row_id'].isin(pred_map)
    submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
sub_path = os.path.join(SUB_DIR, f'blend_optimal_w{overall_best_w:.2f}_v4.csv')
submission.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} (mean={submission["prediction"].mean():.4f})')

# 2. Per-line optimal weights
submission = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    glm_pred = np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01)
    glm_pred = np.clip(glm_pred, 0.01, 50000) * GLM_TREND_FACTORS[line]
    xgb_pred = np.array([xgb_test_preds[line].get(rid, 0.01) for rid in test_df['row_id']])
    w = best_weights[line]
    blend_pred = w * xgb_pred + (1 - w) * glm_pred
    blend_pred = np.clip(blend_pred, 0.01, 50000)
    pred_map = dict(zip(test_df['row_id'], blend_pred))
    mask = submission['row_id'].isin(pred_map)
    submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
sub_path = os.path.join(SUB_DIR, 'blend_per_line_optimal_v4.csv')
submission.to_csv(sub_path, index=False)
print(f'Saved: {sub_path} (mean={submission["prediction"].mean():.4f})')
print(f'  Per-line weights: {best_weights}')

## Step 6: AI/Human Decision Log

### What AI did
- Loaded processed data from Notebook 1 and XGBoost importance from Notebook 2
- Selected top 25 features by XGBoost importance for GLM (user: "include as many as you can")
- Used target encoding (Bayesian shrinkage, SMOOTHING=30) for ALL categoricals — avoids convergence issues from one-hot encoding
- Standardized numeric features with StandardScaler
- Fitted statsmodels GLM Tweedie (power=1.5) with log(exposure) offset, IRLS, max_iter=100
- Validated on 2024 holdout with Tweedie D2
- Blended 50/50 and 70/30 (XGBoost/GLM) and computed D2 on validation
- Generated 3 submission files: glm-only, blend_5050, blend_7030

### What human decided
- Use XGBoost feature importance for GLM feature selection (user instruction)
- Include as many features as possible without convergence issues (user instruction)
- Target encoding approach (AI recommendation based on v3 lessons)
- 25 features (AI choice, balances inclusion with convergence)

### Key decisions
1. **Target encoding vs one-hot**: Target encoding keeps each categorical as 1 column, avoiding dimensionality explosion that caused convergence failure in v3
2. **25 features**: More than v3 (15), including more signal while maintaining convergence
3. **SMOOTHING=30**: Bayesian shrinkage parameter — higher = more shrinkage toward prior
4. **No hypertuning**: Using fixed parameters for speed

### Future improvements
- Hypothesis testing with p-values per rating factor (p-values already in GLM output)
- Grouping of rating factors (age bands, years licensed bands) to prevent overfitting
- Separate probability and severity GLMs
- Large claims modelling (separate treatment for motor large claims)
- Chi-squared testing for categorical features
- Better feature engineering (interaction terms, polynomial features)